[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C58_HardCase_LongTail_Course/03_triggers/03_active_learning_triggers.ipynb)

# 03 · 主动学习与线上触发策略（不确定性 / 一致性 / 规则 / 偏差审计 / 预算优化）

目标：把「回传什么」做成一个**可度量、可优化、可审计**的系统。
你会亲手造出一条带 ground truth 的合成 TSR 检测流，然后实现三类触发器，
并用它复现本模块最重要的那个结论——
**只回传「模型不确定的」，会系统性地漏掉「模型自信地错了」的那一类。**

本 notebook 你会亲手实现：
1. 合成 TSR 检测流：**5 类样本**（已学会 / 低置信但对 / 低置信且错 / **高置信却错** / **完全漏检**）
2. 三种不确定性度量（**熵 / margin / least-confidence**）+ 它们在多类别下的差异
3. **置信度校准**（温度缩放 + ECE）——不确定性触发器的前提条件
4. **框级 → 图级聚合**（max / mean / top-k / count）与它引入的**场景偏差**
5. **多帧一致性触发**：利用「交通标志是静止刚体，类别必须恒定」这一物理性质
6. 多模型分歧（vote entropy）与规则触发（跟踪丢失 / 阈值震荡 / **与高精地图不符** / 接管急刹）
7. **触发器的 PR 分析**：触发率 vs 精确率 vs 召回率
8. **偏差审计**：分组召回率——证明不确定性触发对「高置信却错」的召回 ≈ 0
9. **带宽预算下的贪心分配**（submodular 覆盖）+ 强制随机基线配额

> 心智模型：**触发器是一个二分类器，必须按二分类器来评估。
> 它有 PR 曲线、有工作点，也有系统性偏差——而这个偏差不会体现在任何模型指标上。**

## 1 · 合成一条带 ground truth 的 TSR 检测流

真实系统里我们不知道哪一帧是 badcase（那正是要找的东西）。
所以这里造一条**已知真值**的流，才能把触发器当成分类器来评估。

五类 track（一个 track = 一个标志由远及近的一次接近过程，8 帧）：

| 组 | 置信度 | 是否正确 | 谁能抓到 |
|---|---|---|---|
| `easy` | 高 | ✅（偶发闪烁） | 不需要抓 |
| `unc_correct` | 低 | ✅ | 不确定性触发（**误报**） |
| `unc_wrong` | 低 | ❌ | 不确定性触发 ✅ |
| **`conf_wrong`** | **高** | **❌（远处认错，近处改口）** | **一致性 / 地图，不确定性抓不到** |
| **`silent_miss`** | — | **完全漏检（没有框）** | **只有地图 / 随机基线** |

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
C, T, N_FRAMES = 12, 8, 9000                 # 12 个类别 / 每个 track 8 帧 / 9000 帧
GROUP_N = {'easy': 1500, 'unc_correct': 120, 'unc_wrong': 60,
           'conf_wrong': 55, 'silent_miss': 35}
DET_P   = {'easy': 0.995, 'unc_correct': 0.90, 'unc_wrong': 0.90,
           'conf_wrong': 0.93, 'silent_miss': 0.0}

def one_prob(pred, runner, p1, p2):
    """构造一条类别概率：pred 拿 p1，次高类 runner 拿 p2，剩下的均分。"""
    v = np.full(C, max(1.0 - p1 - p2, 1e-6) / (C - 2))
    v[pred] = p1; v[runner] = p2
    return v / v.sum()

obs = {k: [] for k in ['track', 't', 'frame', 'group', 'true', 'pred', 'det', 'p1']}
PROBS, TRACK_META = [], []
tid = 0
for g, n in GROUP_N.items():
    for _ in range(n):
        true_c = int(rng.integers(0, C))
        conf_c = int((true_c + rng.integers(1, C)) % C)     # 易混淆类（限速 60 vs 80）
        start = int(rng.integers(0, N_FRAMES - T))
        switch = int(rng.integers(3, 6))                    # conf_wrong 第几帧才改口
        map_has = bool(rng.random() < 0.60)                 # 高精地图只覆盖 60% 的标志
        TRACK_META.append(dict(tid=tid, group=g, true=true_c, map_has=map_has))
        for t in range(T):
            det = bool(rng.random() < DET_P[g])
            if g == 'easy':
                pred = true_c if rng.random() > 0.008 else conf_c   # 0.8% 偶发闪烁
                p1 = min(0.97, 0.72 + 0.03 * t + rng.uniform(0, 0.04))
                p2 = (1 - p1) * rng.uniform(0.3, 0.6)
            elif g == 'unc_correct':
                pred = true_c
                p1 = rng.uniform(0.34, 0.52); p2 = p1 * rng.uniform(0.78, 0.98)
            elif g == 'unc_wrong':
                pred = conf_c if rng.random() > 0.15 else true_c    # 低置信 -> 会闪回
                p1 = rng.uniform(0.34, 0.52); p2 = p1 * rng.uniform(0.78, 0.98)
            elif g == 'conf_wrong':
                pred = conf_c if t < switch else true_c             # **远处自信地错**
                p1 = 0.86 + rng.uniform(0, 0.10); p2 = (1 - p1) * rng.uniform(0.3, 0.6)
            else:                                                   # silent_miss
                pred, p1, p2 = true_c, 0.50, 0.20                   # 占位；det 恒为 False
            runner = conf_c if pred != conf_c else true_c
            PROBS.append(one_prob(pred, runner, p1, p2))
            obs['track'].append(tid); obs['t'].append(t); obs['frame'].append(start + t)
            obs['group'].append(g);   obs['true'].append(true_c); obs['pred'].append(pred)
            obs['det'].append(det);   obs['p1'].append(p1)
        tid += 1

O = {k: np.asarray(v) for k, v in obs.items()}
PROBS = np.asarray(PROBS)
O['bad'] = (~O['det']) | (O['det'] & (O['pred'] != O['true']))     # 漏检 或 认错
N_OBS = len(PROBS)

frame_bad = np.zeros(N_FRAMES, dtype=bool)
np.logical_or.at(frame_bad, O['frame'], O['bad'])
n_box = np.bincount(O['frame'][O['det']], minlength=N_FRAMES)      # 每帧的**已检出**框数

print(f'track {tid} 个 / 观测 {N_OBS} 条 / 帧 {N_FRAMES} 帧，平均每帧 {n_box.mean():.2f} 个框')
print(f'{"组":<14s}{"track 数":>9s}{"观测数":>8s}{"其中是 badcase":>15s}')
for g in GROUP_N:
    m = O['group'] == g
    print(f'{g:<14s}{GROUP_N[g]:>9d}{m.sum():>8d}{O["bad"][m].mean():>14.0%}')
print(f'\n**帧级 badcase 率 = {frame_bad.mean():.1%}**（真实车队里要低两三个数量级，'
      f'这里放大是为了统计稳定）')
assert 0.05 < frame_bad.mean() < 0.35
assert O['bad'][O['group'] == 'silent_miss'].all(), '漏检组每一条观测都是 badcase'
assert O['bad'][O['group'] == 'unc_correct'].mean() < 0.15, '低置信但正确的组基本不是 badcase'
print('✅ 数据就位：**有真值**，所以可以把触发器当二分类器来评估')

## 2 · 三种不确定性度量：熵 / margin / least-confidence

$H(p)=-\sum_c p_c\log p_c$，  $M(p)=p_{(1)}-p_{(2)}$，  $LC(p)=1-p_{(1)}$

在**类别多**的任务（TSR 有上百个细分类）上三者差别很大：
熵会被长尾小概率拉高，而 margin 只看前两名——正对应 TSR「限速 60 vs 80」这种成对混淆结构。

In [ ]:
def entropy(P):     return -(P * np.log(P + 1e-12)).sum(-1)
def margin(P):
    s = np.sort(P, axis=-1); return s[..., -1] - s[..., -2]
def least_conf(P):  return 1.0 - P.max(-1)

# 三个「不确定性分数」（越大越可疑）
U = {'entropy': entropy(PROBS), 'one_minus_margin': 1.0 - margin(PROBS),
     'least_conf': least_conf(PROBS)}

print(f'{"组":<14s}' + ''.join(f'{k:>20s}' for k in U))
for g in GROUP_N:
    m = (O['group'] == g) & O['det']
    if m.sum() == 0:
        continue
    print(f'{g:<14s}' + ''.join(f'{U[k][m].mean():>20.3f}' for k in U))

det = O['det']
u_unc  = U['one_minus_margin'][(O['group'] == 'unc_wrong') & det].mean()
u_conf = U['one_minus_margin'][(O['group'] == 'conf_wrong') & det].mean()
u_easy = U['one_minus_margin'][(O['group'] == 'easy') & det].mean()
print(f'\n⚠️  **conf_wrong 的不确定性（{u_conf:.3f}）比 easy（{u_easy:.3f}）还低** —— ')
print(f'    它们「自信地错了」，而不确定性触发器看到的分数和「已经学会」的样本没有区别。')
print(f'    unc_wrong 的不确定性是 {u_unc:.3f}，高出一个量级。')
assert u_unc > 5 * u_conf, '低置信错例的不确定性应远高于高置信错例'
assert u_conf < u_easy * 1.2, 'conf_wrong 的不确定性与 easy 处于同一水平 —— 这就是盲区'

# 熵 vs margin：类别数多时，熵会被「长尾小概率」污染
tight = one_prob(0, 1, 0.45, 0.43)                  # 真正的二选一混淆
diffuse = np.full(C, 1 / C); diffuse[0] = 0.45
diffuse[1:] = (1 - 0.45) / (C - 1)                  # top-1 同样是 0.45，但剩余均匀弥散
print(f'\n{"分布":<26s}{"top1":>7s}{"熵":>9s}{"1-margin":>11s}')
for nm, v in [('二选一混淆 (0.45/0.43)', tight), ('弥散 (0.45 + 均匀残余)', diffuse)]:
    print(f'{nm:<24s}{v.max():>7.2f}{entropy(v[None])[0]:>9.3f}{1 - margin(v[None])[0]:>11.3f}')
assert entropy(diffuse[None])[0] > entropy(tight[None])[0], '弥散分布的熵更高'
assert margin(tight[None])[0] < margin(diffuse[None])[0], '但真正混淆的是「二选一」那个'
print('\n⚠️  两个分布的 top-1 都是 0.45，但**熵把弥散那个排得更靠前**，')
print('    而 TSR 真正要抓的是「二选一混淆」。**类别数越多，熵越不可靠。**')
print('✅ TSR 场景下 margin 通常优于熵；least-confidence 最便宜，适合车端第一层粗筛。')

In [ ]:
# ── 校准：不确定性触发器的前提条件 ──
# 现代网络的 softmax 是**系统性过自信**的：0.9 的置信度对应的真实正确率往往低得多
def ece(conf, correct, n_bins=10):
    """Expected Calibration Error：分箱后 |平均置信度 - 平均正确率| 的加权和。"""
    edges = np.linspace(0, 1, n_bins + 1)
    e, n = 0.0, len(conf)
    for i in range(n_bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.sum() == 0:
            continue
        e += m.sum() / n * abs(conf[m].mean() - correct[m].mean())
    return float(e)

def temp_scale(P, temp):
    z = np.log(P + 1e-12) / temp
    z -= z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

# 造一个**典型的过自信模型**：真实正确概率 a，但网络报告 conf = a^0.45 > a
r7 = np.random.default_rng(33)
a_true = r7.beta(5.0, 1.5, 6000)                        # 每个样本的真实正确概率
corr = (r7.random(6000) < a_true).astype(float)         # 实际对错
conf_rep = a_true ** 0.45                               # **过自信**：把概率往 1 推
Pd = np.stack([conf_rep, 1.0 - conf_rep], axis=1)       # 化成二类分布便于温度缩放
conf0 = Pd.max(-1)

print(f'{"置信度区间":<14s}{"样本数":>8s}{"平均置信度":>12s}{"真实正确率":>12s}{"差距":>9s}')
for lo, hi in [(0.5, 0.7), (0.7, 0.85), (0.85, 0.95), (0.95, 1.0)]:
    m = (conf0 > lo) & (conf0 <= hi)
    if m.sum() == 0:
        continue
    print(f'{f"({lo:.2f},{hi:.2f}]":<14s}{m.sum():>8d}{conf0[m].mean():>12.3f}'
          f'{corr[m].mean():>12.3f}{conf0[m].mean()-corr[m].mean():>9.3f}')

grid = np.linspace(0.5, 4.0, 36)
eces = [ece(temp_scale(Pd, t).max(-1), corr) for t in grid]
t_best = float(grid[int(np.argmin(eces))])
print(f'\nECE(T=1.0) = {ece(conf0, corr):.4f}   ->   ECE(T={t_best:.2f}) = {min(eces):.4f}')
assert (conf0 - corr).mean() > 0.05, '构造出来的就是一个过自信模型'
assert min(eces) < 0.5 * ece(conf0, corr), '温度缩放应大幅降低 ECE'
assert t_best > 1.0, '过自信的模型需要 T>1 把分布软化'
print('✅ 温度缩放（一个标量参数，在验证集上拟合）几乎零成本地修正了过自信。')
print('⚠️  未校准的模型上，「熵 > 1.7」这类绝对阈值是**没有意义**的，')
print('    而且校准误差在不同尺寸/光照桶上还不一样（小目标通常更过自信）')
print('    -> **触发器阈值必须分桶设定，或者干脆按分位数设阈**。')

## 3 · 框级 → 图级聚合：一个隐藏的场景偏差源

回传决策是**整帧**的，但不确定性定义在**框**上。
这一步聚合看似是细节，实际决定了触发器会系统性偏向什么样的场景。

In [ ]:
def aggregate(frame_of, score, n_frames, how='max', k=3, thr=0.5):
    """把框级分数聚合到帧级。frame_of: 每个框所属帧；未检出的框不参与。"""
    out = np.zeros(n_frames)
    order = np.argsort(frame_of, kind='mergesort')
    f_sorted, s_sorted = frame_of[order], score[order]
    bounds = np.searchsorted(f_sorted, np.arange(n_frames + 1))
    for f in range(n_frames):
        s = s_sorted[bounds[f]:bounds[f + 1]]
        if s.size == 0:
            continue
        if how == 'max':     out[f] = s.max()
        elif how == 'mean':  out[f] = s.mean()
        elif how == 'topk':  out[f] = np.sort(s)[-min(k, s.size):].mean()
        elif how == 'count': out[f] = (s > thr).sum()
        elif how == 'count_norm': out[f] = (s > thr).sum() / s.size
    return out

fo, sc = O['frame'][det], U['one_minus_margin'][det]
AGG = {h: aggregate(fo, sc, N_FRAMES, how=h) for h in
       ['max', 'mean', 'topk', 'count', 'count_norm']}

BUDGET = 0.05                                     # 假设只能回传 5% 的帧
print(f'{"聚合算子":<14s}{"触发帧平均框数":>16s}{"全体平均框数":>14s}{"badcase 精确率":>16s}')
has_box = n_box > 0
for h, v in AGG.items():
    kf = int(BUDGET * N_FRAMES)
    sel = np.argsort(-v, kind='mergesort')[:kf]
    print(f'{h:<14s}{n_box[sel].mean():>16.2f}{n_box[has_box].mean():>14.2f}'
          f'{frame_bad[sel].mean():>16.1%}')

kf = int(BUDGET * N_FRAMES)
nb = {h: n_box[np.argsort(-v, kind='mergesort')[:kf]].mean() for h, v in AGG.items()}
assert nb['mean'] < nb['max'], 'mean 聚合偏向框少的帧'
assert nb['count'] > nb['max'], 'count 聚合偏向框多的帧（拥挤路口）'
print(f'\n⚠️  **mean 偏向框少的简单帧**（{nb["mean"]:.2f} 个框 vs 全体 {n_box[has_box].mean():.2f}），')
print(f'    **count 偏向框多的拥挤路口**（{nb["count"]:.2f} 个框）。')
print('    两者都是**由聚合算子引入、与模型无关的采样偏差** —— ')
print('    它不体现在任何模型指标上，却会让某类场景的问题永远发现不了。')
print('✅ 默认用 **max**（TSR 关心「有没有一个标志被认错」，不关心平均质量）；')
print('   要抗单帧噪声就用 top-k mean；count 必须归一化才能用。')

## 4 · 多帧一致性：TSR 的杀手锏

交通标志是**静止刚体，类别在时间上恒定**。
所以同一个 track 上出现类别跳变 = **一个无需标注即可确认的错误**（至少有一帧一定是错的）。

这是 TSR 独有的红利：行人/车辆的属性会真的变化，没有这个性质。

In [ ]:
# 组织成 track -> 帧序列
tr_pred = {}; tr_frames = {}; tr_det = {}
for i in range(N_OBS):
    tr_pred.setdefault(O['track'][i], []).append(O['pred'][i])
    tr_frames.setdefault(O['track'][i], []).append(O['frame'][i])
    tr_det.setdefault(O['track'][i], []).append(O['det'][i])

def temporal_stats(preds, dets):
    """只看**已检出**帧上的类别序列。"""
    seq = [p for p, d in zip(preds, dets) if d]
    if len(seq) < 2:
        return dict(flip=0, mode_frac=1.0, n=len(seq))
    flip = sum(1 for a, b in zip(seq, seq[1:]) if a != b)
    vals, cnt = np.unique(seq, return_counts=True)
    return dict(flip=int(flip), mode_frac=float(cnt.max() / len(seq)), n=len(seq))

TSTAT = {t: temporal_stats(tr_pred[t], tr_det[t]) for t in tr_pred}
g_of = {m['tid']: m['group'] for m in TRACK_META}

print(f'{"组":<14s}{"平均跳变次数":>14s}{"众数占比":>11s}{"至少跳变一次的 track":>22s}')
for g in GROUP_N:
    ts = [TSTAT[t] for t in TSTAT if g_of[t] == g]
    if not ts:
        continue
    print(f'{g:<14s}{np.mean([x["flip"] for x in ts]):>14.2f}'
          f'{np.mean([x["mode_frac"] for x in ts]):>11.2f}'
          f'{np.mean([x["flip"] > 0 for x in ts]):>21.0%}')

rec_conf = np.mean([TSTAT[t]['flip'] > 0 for t in TSTAT if g_of[t] == 'conf_wrong'])
rec_easy = np.mean([TSTAT[t]['flip'] > 0 for t in TSTAT if g_of[t] == 'easy'])
assert rec_conf > 0.90, '多帧一致性应几乎抓住全部「高置信却错」的 track'
assert rec_easy < 0.15, '正常 track 不应频繁误触发'
print(f'\n✅ **多帧类别跳变对 conf_wrong 的召回 = {rec_conf:.0%}** —— ')
print(f'   而这一组的单帧不确定性和 easy 完全无法区分（上一节已验证）。')
print(f'   误报率（easy 组）只有 {rec_easy:.0%}，且成本几乎为零（跟踪本来就要跑）。')

# 帧级一致性分数：该帧所属 track 的「不一致程度」
temporal_score = np.zeros(N_FRAMES)
for t, st in TSTAT.items():
    s = 1.0 - st['mode_frac']
    for f, d in zip(tr_frames[t], tr_det[t]):
        if d:
            temporal_score[f] = max(temporal_score[f], s)

# ── 时序反标：用近处高置信结果反标远处困难帧（免费的困难样本）──
gain = 0
for t, st in TSTAT.items():
    seq = [(p, d) for p, d in zip(tr_pred[t], tr_det[t])]
    if st['flip'] > 0 and st['n'] >= 4:
        gain += sum(1 for k, (p, d) in enumerate(seq) if d and k < 4)   # 远处的困难帧
print(f'\n✅ 附带红利：跳变 track 里可用「近处结果」反标出 {gain} 帧远距离小目标样本，')
print('   **零标注成本**，而这正是 TSR 最缺的数据类型（track-level label propagation）。')
print('⚠️  两个失效条件：① 跟踪 ID switch（两个相邻标志被跟成一个）→ 要先验证 track 纯度；')
print('    ② **可变电子牌的类别是真的会变的** → 必须从「类别恒定」假设里排除。')

## 5 · 多模型分歧与规则触发

一致性触发的另一条路：K 个独立模型投票。
规则触发则完全不看模型的概率，只看**系统级矛盾**——最便宜、最可解释，也最容易被低估。

In [ ]:
# ── 多模型分歧（query-by-committee）──
K_MODEL = 5
r5 = np.random.default_rng(21)
AGREE = {'easy': 0.97, 'unc_correct': 0.55, 'unc_wrong': 0.55,
         'conf_wrong': 0.58, 'silent_miss': 0.5}
votes = np.zeros((N_OBS, K_MODEL), dtype=int)
for i in range(N_OBS):
    pa = AGREE[O['group'][i]]
    if O['group'][i] == 'conf_wrong' and O['pred'][i] == O['true'][i]:
        pa = 0.95                                   # 近处已改口 -> 模型也都同意了
    for m in range(K_MODEL):
        votes[i, m] = O['pred'][i] if r5.random() < pa else int(O['true'][i])

def vote_entropy(v, n_cls=C):
    cnt = np.bincount(v, minlength=n_cls) / len(v)
    return float(-(cnt[cnt > 0] * np.log(cnt[cnt > 0])).sum())

VE = np.array([vote_entropy(votes[i]) for i in range(N_OBS)])
print(f'{"组":<14s}{"平均 vote entropy":>20s}')
for g in GROUP_N:
    m = (O['group'] == g) & det
    if m.sum():
        print(f'{g:<14s}{VE[m].mean():>20.3f}')
assert VE[(O['group'] == 'conf_wrong') & det].mean() > 2.5 * VE[(O['group'] == 'easy') & det].mean()
print('✅ **多模型分歧能看见「自信地错」**：单模型很确定，但模型之间对不上。')
print('⚠️  代价：车端跑不动 K 个模型 —— 通常只在云端影子复检时用。')

# ── 规则触发（几乎零算力）──
rule = {k: np.zeros(N_FRAMES, dtype=bool) for k in
        ['track_lost', 'osc', 'map_mismatch', 'event']}

# ① 跟踪丢失：检出后中断
for t in tr_pred:
    d = tr_det[t]; fs = tr_frames[t]
    for k in range(1, len(d)):
        if d[k - 1] and not d[k]:
            rule['track_lost'][fs[k]] = True

# ② 阈值附近震荡：track 的 top-1 分数反复穿越工作点 0.5
TAU = 0.50
tr_idx = {}                                       # track -> 按时间排好序的观测下标
for i in np.argsort(O['track'] * (T + 1) + O['t'], kind='mergesort'):
    tr_idx.setdefault(int(O['track'][i]), []).append(int(i))
for t, sel in tr_idx.items():
    sel = np.asarray(sel)
    s = O['p1'][sel][O['det'][sel]]
    if s.size >= 2 and int(((s[:-1] - TAU) * (s[1:] - TAU) < 0).sum()) >= 2:
        for i in sel:                             # 只标记真正贴着工作点的那几帧
            if O['det'][i] and abs(O['p1'][i] - TAU) < 0.05:
                rule['osc'][O['frame'][i]] = True

# ③ 与高精地图不符（地图只覆盖 60% 的标志）
map_has = {m['tid']: m['map_has'] for m in TRACK_META}
for i in range(N_OBS):
    if map_has[O['track'][i]] and O['bad'][i]:
        rule['map_mismatch'][O['frame'][i]] = True

# ④ 接管 / 急刹：随机事件，但在 badcase 帧附近概率高 4 倍
r6 = np.random.default_rng(5)
base = np.where(frame_bad, 0.016, 0.004)
rule['event'] = r6.random(N_FRAMES) < base

print(f'\n{"规则触发器":<16s}{"触发率":>10s}{"精确率":>10s}{"召回率":>10s}')
for k, v in rule.items():
    prec = frame_bad[v].mean() if v.sum() else 0.0
    rec = (v & frame_bad).sum() / frame_bad.sum()
    print(f'{k:<16s}{v.mean():>10.2%}{prec:>10.1%}{rec:>10.1%}')
assert frame_bad[rule['map_mismatch']].mean() > 0.95, '地图不符用的是**模型之外的外部真值**，精确率应极高'
assert frame_bad[rule['event']].mean() > frame_bad.mean(), '事件触发的精确率应高于基线'
print('\n✅ **「与高精地图不符」精确率最高**，因为它用的是模型之外的外部真值 ——')
print('   它同时覆盖「自信地错」与「完全漏检」两个不确定性触发器的结构性盲区。')
print('⚠️  但地图只覆盖 60% 的标志，且**地图本身会过期**（施工/限速调整）——')
print('    触发的可能是「地图该更新了」，这同样有价值但要分开处理。')

## 6 · 把触发器当分类器来评：PR 分析

三个量：**触发率**（成本）、**精确率**（回传的里面有多少值得标）、
**召回率**（真 badcase 抓到了多少）。
注意召回率的分母必须来自**无偏的随机基线采样**，否则根本算不出来。

In [ ]:
SCORES = {
    'unc_margin':  AGG['max'],
    'unc_entropy': aggregate(fo, entropy(PROBS)[det], N_FRAMES, how='max'),
    'disagree':    aggregate(fo, VE[det], N_FRAMES, how='max'),
    'temporal':    temporal_score,
    'map_mismatch': rule['map_mismatch'].astype(float),
    'track_lost':  rule['track_lost'].astype(float),
    'osc':         rule['osc'].astype(float),
    'event':       rule['event'].astype(float),
    'random':      np.random.default_rng(99).random(N_FRAMES),
}

def trigger_at_rate(score, is_bad, rate):
    """取分数最高的 rate 比例的帧（分数为 0 的不算触发），返回工作点指标。"""
    kf = int(round(rate * len(score)))
    order = np.argsort(-score, kind='mergesort')
    sel = order[:kf]
    sel = sel[score[sel] > 0]
    fired = np.zeros(len(score), dtype=bool); fired[sel] = True
    prec = is_bad[fired].mean() if fired.sum() else 0.0
    return dict(rate=fired.mean(), precision=float(prec),
                recall=float((fired & is_bad).sum() / is_bad.sum()), mask=fired)

RATES = [0.02, 0.05, 0.10]
print(f'{"触发器":<15s}' + ''.join(f'{"rate=%.0f%%" % (100*r):>22s}' for r in RATES))
print(f'{"":<15s}' + ''.join(f'{"精确率 / 召回率":>22s}' for _ in RATES))
for k, s in SCORES.items():
    line = f'{k:<15s}'
    for r in RATES:
        mm = trigger_at_rate(s, frame_bad, r)
        cell = '%.0f%% / %.0f%%' % (100 * mm['precision'], 100 * mm['recall'])
        line += f'{cell:>22s}'
    print(line)

base_rate = frame_bad.mean()
m_rand = trigger_at_rate(SCORES['random'], frame_bad, 0.05)
m_temp = trigger_at_rate(SCORES['temporal'], frame_bad, 0.05)
assert abs(m_rand['precision'] - base_rate) < 0.04, '随机基线的精确率 ≈ 基线 badcase 率'
assert m_temp['precision'] > 3 * base_rate, '一致性触发的精确率应远高于随机'
print(f'\n基线 badcase 率 = {base_rate:.1%}（= 随机采样的精确率 {m_rand["precision"]:.1%}）')
print('✅ 精确率必须与**基线 badcase 率**比，而不是看绝对值。')
print('⚠️  召回率的分母是「所有真 badcase」—— 只有靠随机基线全标才能得到。')
print('    只在「被触发的样本」里算召回，得到的永远是 100%，毫无意义。')

## 7 · 偏差审计：不确定性触发器的系统性盲区

上面看到的都是**总体**指标。现在按组拆开看——
这一步会暴露一个总体指标完全掩盖的事实。

In [ ]:
# 每一帧属于哪些组（一帧可能有多个 track）
GROUPS = list(GROUP_N)
frame_group = {g: np.zeros(N_FRAMES, dtype=bool) for g in GROUPS}
for i in range(N_OBS):
    if O['bad'][i]:
        frame_group[O['group'][i]][O['frame'][i]] = True

def bias_audit(fired, tag=''):
    row = {}
    for g in GROUPS:
        m = frame_group[g]
        row[g] = float((fired & m).sum() / m.sum()) if m.sum() else float('nan')
    row['ALL'] = float((fired & frame_bad).sum() / frame_bad.sum())
    return row

def fire_set(k, rate):
    """连续分数 -> 取 top-rate；二值规则 -> 用它的**自然触发集合**（不人为截断）。"""
    s = SCORES[k]
    if set(np.unique(s).tolist()) <= {0.0, 1.0}:
        return s > 0
    return trigger_at_rate(s, frame_bad, rate)['mask']

RATE = 0.10
print(f'连续分数统一取 top-{RATE:.0%}，二值规则用自然触发率，看**各组 badcase 的召回率**：\n')
print(f'{"触发器":<15s}{"触发率":>8s}' + ''.join(f'{g:>13s}' for g in GROUPS) + f'{"总体":>8s}')
audits = {}
for k in ['unc_margin', 'unc_entropy', 'temporal', 'disagree', 'map_mismatch',
          'track_lost', 'random']:
    fired = fire_set(k, RATE)
    a = bias_audit(fired); audits[k] = a
    print(f'{k:<15s}{fired.mean():>8.1%}' + ''.join(f'{a[g]:>13.0%}' for g in GROUPS)
          + f'{a["ALL"]:>8.0%}')

# 判据要与**随机基线**比：lift = 该组召回 / 随机基线在该组的召回
# lift ≈ 1 意味着「这个触发器在该组上没有提供任何信息」
R0 = audits['random']
lift = {k: {g: audits[k][g] / R0[g] for g in list(GROUPS) + ['ALL']} for k in audits}
print(f'\n相对随机基线的 **lift**（≈1 = 毫无信息）：\n')
print(f'{"触发器":<15s}' + ''.join(f'{g:>13s}' for g in GROUPS) + f'{"总体":>8s}')
for k in ['unc_margin', 'temporal', 'disagree', 'map_mismatch']:
    print(f'{k:<15s}' + ''.join(f'{lift[k][g]:>12.1f}×' for g in GROUPS)
          + f'{lift[k]["ALL"]:>7.1f}×')

assert lift['unc_margin']['ALL'] > 1.8,          '不确定性触发在**总体**上确实有效'
assert lift['unc_margin']['unc_wrong'] > 4.0,    '它在低置信错例上很强'
assert lift['unc_margin']['conf_wrong'] < 1.2,   '但对「高置信却错」不比随机采样更好 —— 盲区①'
assert lift['unc_margin']['silent_miss'] < 2.2,  '对「完全漏检」同样近乎无信息 —— 盲区②'
assert lift['temporal']['conf_wrong'] > 5.0,     '一致性触发覆盖盲区①'
assert lift['map_mismatch']['silent_miss'] > 5.0, '地图先验覆盖盲区②'
assert audits['temporal']['conf_wrong'] > 0.60 and audits['map_mismatch']['silent_miss'] > 0.30
print(f'\n⚠️  不确定性触发的**总体** lift = {lift["unc_margin"]["ALL"]:.1f}×（看起来很不错），')
print(f'    在 unc_wrong 上高达 {lift["unc_margin"]["unc_wrong"]:.1f}×；')
print(f'    但在 conf_wrong 上只有 {lift["unc_margin"]["conf_wrong"]:.1f}×、'
      f'silent_miss 上 {lift["unc_margin"]["silent_miss"]:.1f}× —— **与随机采样同一量级**。')
print('    （silent_miss 上那点残余 lift 只是「同一帧里还有别的 track」的巧合共现，')
print('     不是触发器真的看见了漏检。）')
print('    **总体指标完全掩盖了这两个结构性盲区。**')
print('\n两个盲区的成因不同：')
print('  ① **自信地错**：模型自报的不确定性低 -> 按行切（低置信）永远切不到它（它在高置信行）')
print('  ② **完全漏检**：没有检出就没有框，也就没有任何分数可打')
print('\n✅ 对策：一致性触发（多帧/多模型）覆盖 ①，规则触发（地图/跟踪丢失）覆盖 ②，')
print('   **随机基线覆盖「未知的未知」**——它是唯一不经过模型判断的数据通路。')
print(f'   随机基线在每一组上的召回都恰好 ≈ 触发率 {RATE:.0%}，**无偏但低效** —— ')
print('   它的价值不在效率，而在于它是唯一能算出「真实召回率分母」的东西。')

## 8 · 带宽预算下的贪心分配

$\max_{\tau}\;\big|\bigcup_t S_t(\tau_t)\cap\mathcal{B}\big|$  s.t.  $\big|\bigcup_t S_t(\tau_t)\big|\le B\cdot N$

这是一个**集合覆盖（submodular）最大化**问题，贪心有 $1-1/e\approx63\%$ 的近似保证。

In [ ]:
def greedy_allocate(scores, is_bad, budget_frames, step=30, exclude=()):
    """贪心：每轮挑「每花一帧预算能新抓到最多 badcase」的那个触发器，取它接下来的 step 帧。"""
    names = [k for k in scores if k not in exclude]
    order = {k: np.argsort(-scores[k], kind='mergesort') for k in names}
    ptr = {k: 0 for k in names}
    chosen = np.zeros(len(is_bad), dtype=bool)
    alloc = {k: 0 for k in names}
    while chosen.sum() < budget_frames:
        best = None
        for k in names:
            take, i, o = [], ptr[k], order[k]
            while i < len(o) and len(take) < step:
                if not chosen[o[i]] and scores[k][o[i]] > 0:
                    take.append(o[i])
                i += 1
            if not take:
                continue
            gain = is_bad[take].sum() / len(take)
            if best is None or gain > best[1]:
                best = (k, gain, take, i)
        if best is None:
            break
        k, _, take, newptr = best
        room = int(budget_frames - chosen.sum())
        take = take[:room]
        chosen[take] = True; ptr[k] = newptr; alloc[k] += len(take)
    return chosen, alloc

B = 0.15                                     # 预算要设在「单一触发器已经不够用」的区间才有意义
budget = int(B * N_FRAMES)
RESULTS = {}

# 策略 A：全部预算给单一最好的触发器
best_single, best_rec = None, -1
for k in ['unc_margin', 'temporal', 'disagree', 'map_mismatch']:
    m = trigger_at_rate(SCORES[k], frame_bad, B)
    if m['recall'] > best_rec:
        best_single, best_rec, RESULTS['A 单一最优触发器'] = k, m['recall'], m['mask']

# 策略 B：贪心多触发器组合（不含随机）
mask_B, alloc_B = greedy_allocate(SCORES, frame_bad, budget, exclude=('random',))
RESULTS['B 贪心组合'] = mask_B

# 策略 C：组合 + 强制 15% 随机基线配额（对抗触发器偏差）
rand_share = 0.15
n_rand = int(rand_share * budget)
rand_sel = np.random.default_rng(7).permutation(N_FRAMES)[:n_rand]
mask_C = np.zeros(N_FRAMES, dtype=bool); mask_C[rand_sel] = True
extra, alloc_C = greedy_allocate(SCORES, frame_bad, budget - n_rand, exclude=('random',))
mask_C |= extra
RESULTS['C 组合 + 15% 随机基线'] = mask_C

print(f'预算 = {B:.0%} 的帧（{budget} 帧），最优单一触发器 = {best_single}\n')
print(f'{"策略":<26s}{"实际触发率":>12s}{"精确率":>9s}{"总体召回":>10s}'
      + ''.join(f'{g:>13s}' for g in ['conf_wrong', 'silent_miss']))
for nm, msk in RESULTS.items():
    a = bias_audit(msk)
    print(f'{nm:<24s}{msk.mean():>12.2%}{frame_bad[msk].mean():>9.1%}{a["ALL"]:>10.1%}'
          + ''.join(f'{a[g]:>13.0%}' for g in ['conf_wrong', 'silent_miss']))

rec_A = bias_audit(RESULTS['A 单一最优触发器'])['ALL']
rec_B = bias_audit(RESULTS['B 贪心组合'])['ALL']
rec_C = bias_audit(RESULTS['C 组合 + 15% 随机基线'])
assert rec_B > rec_A + 0.05, '多触发器组合应显著优于单一触发器（它们覆盖不同盲区）'
assert rec_C['ALL'] <= rec_B + 1e-9, '留 15% 给随机基线，总召回不会更高 —— 这是刻意付出的代价'
assert rec_C['ALL'] > 0.85 * rec_B, '但这个代价应该很小'
print(f'\n预算分配（策略 B）: ' + ', '.join(f'{k}={v}' for k, v in alloc_B.items() if v))
print(f'\n✅ 贪心组合把总召回从 {rec_A:.1%} 提到 {rec_B:.1%} —— 因为不同触发器覆盖不同盲区，')
print('   而 submodular 覆盖的贪心解有 1-1/e ≈ 63% 的近似保证。')
print(f'✅ 策略 C 留出 15% 预算给随机基线，总召回只从 {rec_B:.1%} 降到 {rec_C["ALL"]:.1%}'
      f'（-{100*(rec_B-rec_C["ALL"]):.1f} pp）——')
print('   **一份极其便宜的保险**：换来一条不经过模型判断的数据通路，')
print('   它是唯一能发现「未知的未知」、也是唯一能算出真实召回率分母的东西。')
print('⚠️  注意精确率从 %.0f%% 降到 %.0f%% —— 随机基线本来就低效，'
      % (100 * frame_bad[mask_B].mean(), 100 * frame_bad[mask_C].mean()))
print('    它的价值不在效率，而在**无偏**。')
print('✅ 量产配置 = 车端便宜触发器粗筛 -> 云端大模型精筛 -> 按场景桶配额 -> 15% 随机基线。')

## ✏️ 练习 1：图级聚合与工作点

实现 `image_uncertainty(box_scores, how, k=3, thr=0.5)`：输入**一帧内**所有框的分数
（可能为空数组），返回该帧的标量分数。支持 `'max' / 'mean' / 'topk' / 'count_norm'`。
空数组一律返回 `0.0`。`count_norm` = 超过 `thr` 的框数 / 总框数。

In [ ]:
def image_uncertainty(box_scores, how='max', k=3, thr=0.5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
s = np.array([0.1, 0.6, 0.9, 0.4])
assert abs(image_uncertainty(s, 'max') - 0.9) < 1e-12
assert abs(image_uncertainty(s, 'mean') - 0.5) < 1e-12
assert abs(image_uncertainty(s, 'topk', k=2) - 0.75) < 1e-12
assert abs(image_uncertainty(s, 'count_norm', thr=0.5) - 0.5) < 1e-12
assert image_uncertainty(np.array([]), 'max') == 0.0
assert image_uncertainty(np.array([]), 'mean') == 0.0
assert abs(image_uncertainty(s, 'topk', k=99) - s.mean()) < 1e-12, 'k 大于框数时退化为 mean'
# 在真实数据上与向量化实现对齐
fo_list = {}
for i in np.where(det)[0]:
    fo_list.setdefault(O['frame'][i], []).append(U['one_minus_margin'][i])
for f in list(fo_list)[:200]:
    assert abs(image_uncertainty(np.array(fo_list[f]), 'max') - AGG['max'][f]) < 1e-12
print('✅ 练习 1 通过：**默认用 max**；mean 偏向框少的帧，count 必须归一化')

## ✏️ 练习 2：多帧一致性触发器

实现 `temporal_trigger(pred_seq, det_seq, min_obs=3)`，只统计**已检出**的帧，返回 dict：

- `'n'`：有效观测数
- `'flip'`：相邻类别变化次数
- `'mode_frac'`：众数类别占比
- `'fire'`：`n >= min_obs` **且** `flip >= 1` 时为 True（观测太少不下结论）

In [ ]:
def temporal_trigger(pred_seq, det_seq, min_obs=3):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r = temporal_trigger([3,3,3,3], [True]*4)
assert r['n'] == 4 and r['flip'] == 0 and abs(r['mode_frac'] - 1.0) < 1e-12 and not r['fire']
r = temporal_trigger([5,5,5,3,3], [True]*5)
assert r['flip'] == 1 and abs(r['mode_frac'] - 0.6) < 1e-12 and r['fire'], r
r = temporal_trigger([5,3,5,3], [True]*4)
assert r['flip'] == 3 and r['fire'], r
r = temporal_trigger([5,3], [True, True])              # 观测太少
assert not r['fire'], '观测数 < min_obs 时不下结论'
r = temporal_trigger([5,9,9,9], [True, False, False, False])
assert r['n'] == 1 and not r['fire'], '未检出的帧不参与'
# 真实数据上：对 conf_wrong 的召回应远高于 easy 的误报
fire = {t: temporal_trigger(tr_pred[t], tr_det[t])['fire'] for t in tr_pred}
rc = np.mean([fire[t] for t in fire if g_of[t] == 'conf_wrong'])
re_ = np.mean([fire[t] for t in fire if g_of[t] == 'easy'])
print(f'conf_wrong 召回 {rc:.0%} | easy 误报 {re_:.0%}')
assert rc > 0.85 and re_ < 0.15
print('✅ 练习 2 通过：**「标志是静止刚体，类别必须恒定」是一个免费的自动标签**')

## ✏️ 练习 3：触发器的 PR 分析与偏差审计

实现 `evaluate_trigger(score, is_bad, group_masks, rate)`，返回 dict：

- `'rate'` / `'precision'` / `'recall'`（整体）
- `'by_group'`：`{组名: 该组 badcase 的召回率}`
- `'lift'`：`precision / is_bad.mean()`（相对随机基线的提升倍数）

选帧规则同正文：取分数最高的 `rate` 比例，且**分数必须 > 0**。

In [ ]:
def evaluate_trigger(score, is_bad, group_masks, rate):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
gm = {g: frame_group[g] for g in GROUPS}
e_rand = evaluate_trigger(SCORES['random'], frame_bad, gm, 0.05)
e_unc  = evaluate_trigger(SCORES['unc_margin'], frame_bad, gm, 0.05)
e_tmp  = evaluate_trigger(SCORES['temporal'], frame_bad, gm, 0.05)
assert abs(e_rand['lift'] - 1.0) < 0.35, '随机基线的 lift ≈ 1'
assert e_tmp['lift'] > 3.0, '一致性触发的 lift 应显著 > 1'
assert set(e_unc['by_group']) == set(GROUPS)
assert e_unc['by_group']['conf_wrong'] < 0.10, '**不确定性触发抓不到「自信地错」**'
assert e_tmp['by_group']['conf_wrong'] > 0.60, '一致性触发能抓到'
assert abs(e_rand['recall'] - 0.05) < 0.03, '随机基线的召回 ≈ 触发率'
print(f'{"触发器":<14s}{"lift":>7s}{"总召回":>9s}{"conf_wrong 召回":>17s}')
for nm, e in [('random', e_rand), ('unc_margin', e_unc), ('temporal', e_tmp)]:
    print(f'{nm:<14s}{e["lift"]:>7.1f}{e["recall"]:>9.0%}{e["by_group"]["conf_wrong"]:>17.0%}')
print('✅ 练习 3 通过：**总体召回会掩盖结构性盲区，必须分组审计**')

## ✏️ 练习 4：带宽预算下的触发器组合

实现 `plan_triggers(scores, is_bad, budget_rate, random_share=0.15, step=30)`：

1. 先把 `budget_rate * N` 帧里的 `random_share` 比例**无条件**分给随机采样（种子固定为 7）
2. 剩余预算用 `greedy_allocate` 在其余触发器上贪心分配（排除 `'random'`）
3. 返回 `{'mask':…, 'alloc':…, 'recall':…, 'precision':…, 'random_frames':…}`

In [ ]:
def plan_triggers(scores, is_bad, budget_rate, random_share=0.15, step=30):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
p0 = plan_triggers(SCORES, frame_bad, 0.05, random_share=0.0)
p15 = plan_triggers(SCORES, frame_bad, 0.05, random_share=0.15)
assert p0['random_frames'] == 0 and p15['random_frames'] == int(0.15 * int(0.05 * N_FRAMES))
assert abs(p0['mask'].mean() - 0.05) < 0.005 and abs(p15['mask'].mean() - 0.05) < 0.006
assert p0['recall'] > p15['recall'], '留随机配额必然牺牲一点总召回'
assert p15['recall'] > 0.75 * p0['recall'], '但代价应可接受'
assert sum(p0['alloc'].values()) > 0 and 'random' not in p0['alloc']
print(f'{"配置":<22s}{"触发率":>9s}{"精确率":>9s}{"总召回":>9s}')
for nm, p in [('无随机基线', p0), ('15% 随机基线', p15)]:
    print(f'{nm:<20s}{p["mask"].mean():>9.2%}{p["precision"]:>9.1%}{p["recall"]:>9.1%}')
print('分配:', {k: v for k, v in p15['alloc'].items() if v})
print('✅ 练习 4 通过：**随机基线是刻意付出的代价** —— 它换来无偏审计与「未知的未知」')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def image_uncertainty(box_scores, how='max', k=3, thr=0.5):
    s = np.asarray(box_scores, dtype=float)
    if s.size == 0:
        return 0.0
    if how == 'max':
        return float(s.max())
    if how == 'mean':
        return float(s.mean())
    if how == 'topk':
        return float(np.sort(s)[-min(k, s.size):].mean())
    if how == 'count_norm':
        return float((s > thr).sum() / s.size)
    raise ValueError(how)

In [ ]:
# 练习 2 参考答案
def temporal_trigger(pred_seq, det_seq, min_obs=3):
    seq = [int(p) for p, d in zip(pred_seq, det_seq) if d]
    n = len(seq)
    if n == 0:
        return dict(n=0, flip=0, mode_frac=1.0, fire=False)
    flip = sum(1 for a, b in zip(seq, seq[1:]) if a != b)
    _, cnt = np.unique(seq, return_counts=True)
    mode_frac = float(cnt.max() / n)
    return dict(n=n, flip=int(flip), mode_frac=mode_frac,
                fire=bool(n >= min_obs and flip >= 1))

In [ ]:
# 练习 3 参考答案
def evaluate_trigger(score, is_bad, group_masks, rate):
    kf = int(round(rate * len(score)))
    order = np.argsort(-np.asarray(score), kind='mergesort')[:kf]
    order = order[np.asarray(score)[order] > 0]
    fired = np.zeros(len(score), dtype=bool); fired[order] = True
    base = is_bad.mean()
    prec = float(is_bad[fired].mean()) if fired.sum() else 0.0
    return dict(rate=float(fired.mean()), precision=prec,
                recall=float((fired & is_bad).sum() / is_bad.sum()),
                lift=prec / base if base > 0 else 0.0,
                by_group={g: (float((fired & m).sum() / m.sum()) if m.sum() else float('nan'))
                          for g, m in group_masks.items()},
                mask=fired)

In [ ]:
# 练习 4 参考答案
def plan_triggers(scores, is_bad, budget_rate, random_share=0.15, step=30):
    n = len(is_bad)
    budget = int(budget_rate * n)
    n_rand = int(random_share * budget)
    mask = np.zeros(n, dtype=bool)
    if n_rand:                                   # ① 无条件的随机基线配额
        mask[np.random.default_rng(7).permutation(n)[:n_rand]] = True
    extra, alloc = greedy_allocate(scores, is_bad, budget - n_rand,
                                   step=step, exclude=('random',))
    mask |= extra                                # ② 剩余预算贪心分配
    return dict(mask=mask, alloc=alloc, random_frames=n_rand,
                precision=float(is_bad[mask].mean()) if mask.sum() else 0.0,
                recall=float((mask & is_bad).sum() / is_bad.sum()))

---
## 🧪 真实工程胶囊：车端触发器配置 + 每日看板 + 上线检查清单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# 车端触发器 · 生产配置模板（可直接改成 yaml / protobuf）
# ══════════════════════════════════════════════════════════════════

# ── 总预算（一切设计的出发点）─────────────────────────────────────
#   10 万辆车 × 1 h/天 × 30 FPS = 1.08e10 帧/天
#   标注预算 5 万帧/天  ->  **允许触发率 ≈ 4.6e-6**
#   ⇒ 触发器 precision 从 5% 提到 20% = 标注效率翻两番
budget: {frames_per_day: 50000, bandwidth_gb_per_car_day: 0.2}

# ── ① 车端触发器（算力 < 1 ms/帧，所以只能用「本来就在跑」的信号）──
triggers:
  - name: temporal_class_flip        # **TSR 杀手锏**：静止刚体，类别必须恒定
    signal: track.pred_class_sequence
    fire_if: "flip_count >= 1 and n_obs >= 3"
    cost: ~0                          # 跟踪本来就要跑
    note: "跳变 = 无需标注即可确认的错误；还能用近处结果反标远处困难帧"
    exclude: ["variable_message_sign"]   # ⚠️ 电子可变牌的类别是**真的会变**的

  - name: track_lost
    fire_if: "detected_then_gap and not out_of_frame"

  - name: map_mismatch               # **精确率最高**：用了模型之外的外部真值
    fire_if: "hdmap.has_sign(loc) and (no_detection or pred_class != map_class)"
    note: "同时覆盖『自信地错』与『完全漏检』两个盲区；但地图会过期 -> 分开处理"

  - name: margin_low                 # margin 优于熵（类别多时熵被长尾污染）
    signal: "1 - (p1 - p2)"
    aggregate: max                    # **默认 max**；mean 偏向框少的帧，count 偏向拥挤路口
    fire_if: "score > quantile(0.999, sliding_window)"   # **按分位数设阈，不用绝对阈值**

  - name: takeover_or_hard_brake
    fire_if: "driver_takeover or a_long < -3.0"
    window: [-3s, +3s]

  - name: stratified_random          # ★★ **不能砍**
    share_of_budget: 0.15
    strata: [geo_region, time_of_day, weather]
    note: "唯一不经过模型判断的通路；也是唯一能算出真实召回率分母的东西"

# ── ② 全局控制 ───────────────────────────────────────────────────
dedup:
  onboard_time_window_s: 30           # 同一 track / 同一场景 N 秒内只触发一次
  cloud: perceptual_hash + embedding_cluster    # 云端内容去重（见模块 04）
quota_by_bucket:                      # **按桶配额**，消除地理/时段偏差
  {night: 0.25, rain: 0.15, urban: 0.30, highway: 0.30}
circuit_breaker:                      # **安全阀**：触发率异常时自动降级
  max_rate_multiplier: 3.0
  action: raise_threshold_then_alert
rollout: {canary_fleet_pct: 1, ramp: [1, 5, 25, 100]}   # 触发器也要灰度

# ── ③ 每日看板（触发率看板与模型指标看板同等重要）─────────────────
#   触发器 | 触发率 | 带宽占比 | 精确率* | 新场景占比 | 与其他触发器的重叠度
#   * 精确率分母 = 回传量，分子 = 复检确认确实是 badcase
#   必看三个趋势：① 触发率漂移（模型/跟踪/地图/环境变了）
#                ② 触发器之间的重叠度（重叠高 = 有冗余，可省预算）
#                ③ **从触发到修复上线的周期时间** <- 数据闭环真正的 KPI

# ── 上线前检查清单 ────────────────────────────────────────────────
#   [ ] 概率**校准**过了吗？（温度缩放 + 分桶 ECE）阈值是按分位数还是绝对值？
#   [ ] 图级聚合用的是 max 吗？（mean/count 会引入场景偏差）
#   [ ] 做了**时间窗去重**吗？（不去重，一段 10s 困难路段能吃掉整天预算）
#   [ ] 做了**分组偏差审计**吗？（conf_wrong / silent_miss 两组召回是多少？）
#   [ ] 留了**随机基线配额**吗？没有它就算不出真实召回率
#   [ ] 有**安全阀**吗？触发率暴涨时能自动降级吗？
#   [ ] 触发器配置能**远程灰度与回滚**吗？还是要等下次 OTA？
#   [ ] 脱敏（人脸/车牌）在回传前完成了吗？脱敏会不会破坏训练数据？
'''
print(RECIPE)
for key in ['temporal_class_flip', 'map_mismatch', 'stratified_random', 'circuit_breaker',
            'quota_by_bucket', 'onboard_time_window_s', 'variable_message_sign',
            'quantile(0.999', '周期时间']:
    assert key in RECIPE, key
print('✅ 配方覆盖：预算推导 / 三类触发器 / 去重 / 桶配额 / 安全阀 / 灰度 / 看板 / 检查清单')

### 小结

- **问题变了**：模块 02 是「已标好的数据里该多看哪些」，这里是「**无限的流式数据里该回传什么**」。
  10 万辆车 × 1h/天 × 30FPS ≈ 1e10 帧/天，标注预算 5 万帧/天 → **触发率必须压到 1e-6 量级**。
  在这个量级下，触发器 precision 从 5% 提到 20%，意味着标注效率翻两番。
- **触发器是一个二分类器**，必须按二分类器评估：触发率（成本）/ 精确率 / 召回率。
  **召回率的分母只能来自无偏的随机基线采样**——否则你算出来的永远是 100%。
- **三类触发器，各自覆盖不同盲区**：
  ① **不确定性**（熵 / margin / least-confidence）——类别多时 **margin 优于熵**；
     **前提是概率已校准**，且阈值应按分位数而非绝对值设定；
  ② **一致性**（多模型分歧 / TTA / 教师-学生 / **多帧不一致**）——
     多帧一致性在 TSR 上是杀手锏：**标志是静止刚体，类别必须恒定**，
     跳变即错误，**零标注成本**，还能用近处结果反标远处困难帧；
  ③ **规则**（跟踪丢失 / 阈值震荡 / **与高精地图不符** / 接管急刹 / 罕见类 / 地理围栏）——
     最便宜、最可解释，且**与模型无关**，所以不会随模型一起变瞎。
- **框级 → 图级的聚合算子是一个隐藏的采样偏差源**：mean 偏向框少的简单帧，
  count 偏向拥挤路口。默认用 **max**，抗噪用 top-k mean。
- **触发器本身会引入偏差**（本模块最深的一条）：只回传「模型不确定的」，会
  **系统性漏掉「模型自信地错了」**（高置信 + 错误）与**完全漏检**（没框就没分数）两类。
  而**总体召回指标完全掩盖这一点**——必须做**分组偏差审计**。
  更糟的是这个盲区**自我强化**：触发器是模型的函数，模型是回传数据的函数。
- **随机基线（10–20% 预算）不能砍**：它是唯一不经过模型判断的通路，
  既能发现「未知的未知」，又提供了触发器评估的无偏分母和模型的无偏验证集。
  它会让总召回略降——**这是刻意付出的代价**。
- **预算分配是 submodular 覆盖问题**，贪心即可（1−1/e 保证）。量产配置 =
  车端便宜触发器粗筛 → 云端大模型精筛 → **按场景桶配额** → 15% 分层随机基线 → **安全阀**。
- **数据闭环真正的 KPI 不是回传量，而是「从发现问题到修复上线的周期时间」。**

下一站：**模块 04 · 大规模挖掘基础设施** —— 数据回传之后：嵌入检索、
场景打标（含用 VLM 自动打标）、去重与多样性采样、标注预算分配、数据版本与血缘。